# Estrategia Bitcoin con clasificación — **sesión 3 horas**

**Referencias:** [BitcoinTradingStrategy.ipynb](https://github.com/tatsath/fin-ml/blob/master/Chapter%206%20-%20Sup.%20Learning%20-%20Classification%20models/CaseStudy3%20-%20Bitcoin%20Trading%20Strategy/BitcoinTradingStrategy.ipynb) · [Classification-MasterTemplate.ipynb](https://github.com/tatsath/fin-ml/blob/master/Chapter%206%20-%20Sup.%20Learning%20-%20Classification%20models/Classification-MasterTemplate.ipynb) · *Machine Learning and Data Science Blueprints for Finance*.

---

## Agenda sugerida (~180 min)

| Bloque | Tiempo | Qué hacemos |
|--------|--------|-------------|
| **A** | 0:00 – 0:25 | Problema, datos, EDA breve |
| **B** | 0:25 – 1:15 | Limpieza, etiqueta SMA, indicadores, correlación |
| **C** | 1:15 – 1:55 | Métricas, hiperparámetros (gráficos), comparar modelos |
| **D** | 1:55 – 2:35 | Grid Search RF, validación, importancia de variables |
| **E** | 2:35 – 3:00 | Backtesting, cierre, tarea opcional (modo libro completo) |

**Antes de empezar:** ejecuta la celda **Configuración de la sesión** (`MODO_CLASE_3H = True` por defecto → menos filas y CV más rápida para terminar en 3 h en Colab).

**Pedagogía:** **una idea por celda** — markdown de interpretación → código corto → siguiente paso. Evita saltar bloques largos sin leer.

**Roles:** en A–B conviene ejecutar **celda a celda**; en C–D puedes usar **Run all** desde la partición train/val si el tiempo apremia.


## Contenido del cuaderno

* [Configuración de la sesión](#cfg)
* [1. Definición del problema](#0)
* [2. Librerías y datos](#1)
* [3. EDA (breve)](#2)
* [4. Preparación de datos](#3)
* [5. Evaluación y modelos](#4)
* [6. Grid Search](#5)
* [7. Modelo final](#6)
* [8. Backtesting](#7)
* [9. Modo completo fin-ml (opcional / tarea)](#8)


<a id='0'></a>
# 1. Definición del problema

## 1.1 Enfoque de la aplicación (Bitcoin / trading)

| Paso | Qué hacemos | Para qué sirve en trading |
|------|-------------|---------------------------|
| 1 | Cargar OHLCV minuto a minuto (Bitstamp) | Tener la serie de precios y liquidez |
| 2 | Limpiar y ordenar en el tiempo | Evitar señales con huecos falsos |
| 3 | Definir **etiqueta** con SMA 10 vs SMA 60 | Traducir “¿estoy alcista de corto plazo?” en 0/1 |
| 4 | Construir **indicadores** (RSI, estocástico, etc.) | Dar al ML pistas de tendencia y momentum |
| 5 | Entrenar clasificadores | Aproximar (y a veces mejorar) la regla SMA |
| 6 | Elegir modelo + hiperparámetros | Balance entre acierto y estabilidad |
| 7 | Evaluar en **validación** (matriz de confusión) | Ver falsas compras vs subidas perdidas |
| 8 | **Backtest** simple | Simular PnL antes de arriesgar capital |

**Señal de referencia (libro / fin-ml):**

- **1 = compra / largo:** SMA₁₀(Close) > SMA₆₀(Close).
- **0 = no largo:** en caso contrario.

El ML **no sustituye** esa idea: aprende una función $f(X_t) \approx \text{signal}_t$ usando muchos indicadores a la vez.

## 1.2 Enfoque de modelos (plantilla maestra de clasificación)

| Paso ML | Sección del notebook | Idea |
|---------|----------------------|------|
| 1 | EDA | Entender datos antes de modelar |
| 2 | Preparación | $Y$ = signal, $X$ = indicadores |
| 3 | Train / validation split | Medir generalización |
| 4 | Validación cruzada | Comparar algoritmos sin mirar el test |
| 5 | Grid Search | Ajustar hiperparámetros del ganador |
| 6 | Modelo final + métricas | Reportar en hold-out |
| 7 | Persistencia + backtest | Cerrar el ciclo “investigación → decisión” |

**Datos:** muestra en GitHub; números del PDF con [Kaggle — Bitstamp minutes](https://www.kaggle.com/mlfinancebook/bitstamp-bicoin-minutes-data).


<a id='1'></a>
# 2. Inicio — librerías y datos
<a id='1.1'></a>
## 2.1 Cargar librerías

Mismas familias que el notebook original: `pandas`, `sklearn`, visualización y (opcional) redes vía `MLPClassifier` — equivalente práctico a la red shallow del master template en Colab sin Keras.


**Paso 2.1a — Detectar Colab**

- **Qué hace:** Comprueba si el notebook corre en Google Colab.
- **Qué mirar en la salida:** Debe imprimir `Colab: True` o `False`.
- **Lectura ML:** En local no hace falta `pip`; en Colab instalamos paquetes si faltan.


In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print('Colab:', IN_COLAB)


**Paso 2.1b — Instalar dependencias (solo Colab)**

- **Qué hace:** Instala `seaborn` y `scikit-learn` si estamos en Colab.
- **Qué mirar en la salida:** Si no estás en Colab, esta celda no instala nada (está bien).


In [ ]:
if IN_COLAB:
    !pip -q install seaborn scikit-learn
else:
    print('Entorno local: usa tu venv / conda.')


**Paso 2.1c — NumPy, Pandas y gráficos**

- **Qué hace:** Importa herramientas para tablas y visualización.
- **Qué mirar en la salida:** Sin errores de importación.
- **Lectura trading (Bitcoin):** Aquí manipularemos precios minuto a minuto (OHLCV).


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas import read_csv, set_option
from pandas.plotting import scatter_matrix
import seaborn as sns


**Paso 2.1d — Scikit-learn — preparación y validación**

- **Qué hace:** Importa partición de datos, CV y escalado.
- **Qué mirar en la salida:** Nombres como `train_test_split`, `GridSearchCV`.
- **Lectura ML:** Estas funciones implementan la plantilla maestra de clasificación.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, KFold, cross_val_score, GridSearchCV, validation_curve,
)


**Paso 2.1e — Scikit-learn — modelos**

- **Qué hace:** Importa los clasificadores que compararemos (como fin-ml).
- **Qué mirar en la salida:** Lista de clases sin error.
- **Lectura ML:** Cada algoritmo es un candidato para aproximar la etiqueta SMA.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier


**Paso 2.1f — Métricas y guardado**

- **Qué hace:** Importa métricas de clasificación y `pickle` para persistir el modelo.
- **Qué mirar en la salida:** Funciones `accuracy_score`, `confusion_matrix`, etc.
- **Lectura trading (Bitcoin):** Precision/recall traducen errores en falsas compras o subidas perdidas.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from pickle import dump, load
print('Librerías listas.')


<a id='cfg'></a>
## Configuración de la sesión (ejecutar primero)

- **`MODO_CLASE_3H = True`:** ~30 000 filas finales, 3 folds, 5 modelos en el benchmark, grid RF reducido → **cabe en ~3 h**.
- **`MODO_CLASE_3H = False`:** réplica del flujo del repo (100 000 filas, 10 folds, 9 modelos) → mejor para **tarea en casa** o sesión extendida.


**Paso 2.2a — Modo docente**

- **Qué hace:** Elige entre sesión de 3 h o réplica completa del repo.
- **Qué mirar en la salida:** `MODO_CLASE_3H = True` para clase; `False` para tarea.
- **Lectura ML:** Menos filas y folds = menos tiempo de cómputo en Colab.


In [ ]:
MODO_CLASE_3H = True
print('Modo:', 'CLASE 3h' if MODO_CLASE_3H else 'COMPLETO fin-ml')


**Paso 2.2b — Tamaño de muestra y CV**

- **Qué hace:** Define cuántas filas finales y cuántos folds usar.
- **Qué mirar en la salida:** Imprime `N_FILAS` y `N_FOLDS`.
- **Lectura trading (Bitcoin):** Modelar solo el tramo reciente enfatiza el régimen de mercado actual (con trade-offs).
- **Lectura ML:** Más folds = estimación más estable pero más lento.


In [ ]:
N_FILAS = 30_000 if MODO_CLASE_3H else 100_000
N_FOLDS = 3 if MODO_CLASE_3H else 10
CV_SEED = 7
print('Filas para modelado:', N_FILAS, '| Folds CV:', N_FOLDS, '| Seed:', CV_SEED)


---
## ⏱ Bloque A · Problema y datos (~25 min)
<a id='1.2'></a>
### 2.2 Cargar datos

> **Nota del libro:** en GitHub la muestra es pequeña por límite de tamaño; los números del PDF se reproducen con el CSV completo en Kaggle.


**Paso A1 — Rutas de datos**

- **Qué hace:** Define archivos locales posibles (Colab, repo clonado, carpeta `data/`).
- **Qué mirar en la salida:** Lista `CANDIDATES` sin ejecutar lectura aún.


In [ ]:
from pathlib import Path

CANDIDATES = [
    Path('data/BitstampData_sample.csv'),
    Path('Ejemplo_Clase/data/BitstampData_sample.csv'),
    Path('/content/ml_clasificacion/Ejemplo_Clase/data/BitstampData_sample.csv'),
]
print('Rutas locales configuradas:', len(CANDIDATES))


**Paso A2 — URLs de respaldo**

- **Qué hace:** Si no hay CSV local, intentará descargar desde GitHub.
- **Qué mirar en la salida:** Lista `URLS` (ml_clasificacion y fin-ml).


In [ ]:
URLS = [
    'https://raw.githubusercontent.com/owoc9103/ml_clasificacion/main/Ejemplo_Clase/data/BitstampData_sample.csv',
    'https://raw.githubusercontent.com/tatsath/fin-ml/master/Chapter%206%20-%20Sup.%20Learning%20-%20Classification%20models/CaseStudy3%20-%20Bitcoin%20Trading%20Strategy/BitstampData_sample.csv',
]


**Paso A3 — Leer CSV**

- **Qué hace:** Carga el primer archivo local encontrado; si no, descarga.
- **Qué mirar en la salida:** Mensaje `Archivo local:` o `Descargado:` y número de filas.
- **Lectura trading (Bitcoin):** Cada fila es un minuto de Bitstamp: base de la estrategia intradía.


In [ ]:
dataset = None
for p in CANDIDATES:
    if p.exists():
        dataset = read_csv(p)
        print('Archivo local:', p)
        break
if dataset is None:
    for url in URLS:
        try:
            dataset = read_csv(url)
            print('Descargado:', url)
            break
        except Exception as exc:
            print('No disponible:', url, exc)

assert dataset is not None, 'Sube data/BitstampData_sample.csv o revisa las URLs'
print('Filas cargadas:', len(dataset))


### Paso a paso — qué es cada columna (Bitcoin)

| Columna | Significado en mercado |
|---------|-------------------------|
| `Open`, `High`, `Low`, `Close` | Precio en ese minuto (vela de 1 min) |
| `Volume_(BTC)` | Cantidad negociada en BTC (liquidez relativa) |
| `Volume_(Currency)` | Volumen en moneda fiat (se elimina más adelante en el repo) |
| `Weighted_Price` | Precio promedio ponderado del minuto |

Cada **fila = un minuto**. La estrategia del libro decide en cada minuto si el régimen es “alcista de corto plazo” (etiqueta) usando medias e indicadores **calculados solo con información hasta ese minuto** (cuidado con *look-ahead* en proyectos propios).


**Paso A4 — Herramientas de visualización en notebook**

- **Qué hace:** Configura `display` y precisión de pandas.
- **Qué mirar en la salida:** `Librerías listas` implícito al ejecutar sin error.


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print
set_option('display.width', 100)
set_option('precision', 3)


**Paso A5 — Dimensiones del dataset**

- **Qué hace:** Muestra filas × columnas.
- **Qué mirar en la salida:** Tupla `(filas, columnas)` — compara con Kaggle completo si haces tarea.


In [ ]:
print('Shape (filas, columnas):', dataset.shape)


**Después de ejecutar:**

- ¿Cuántas filas tienes? En clase usaremos solo las últimas `N_FILAS` minutos.
- *ML:* Menos filas = entrenamiento más rápido pero menos historia.


**Paso A6 — Cola de la serie**

- **Qué hace:** Muestra los últimos 5 minutos cargados.
- **Qué mirar en la salida:** Precios y volúmenes recientes; orden temporal (más nuevo abajo).
- **Lectura trading (Bitcoin):** Verifica que `Close` tenga sentido (no ceros ni saltos absurdos).


In [ ]:
display(dataset.tail(5))


**Paso A7 — Estadísticas descriptivas**

- **Qué hace:** Resumen numérico por columna (`describe`).
- **Qué mirar en la salida:** Columna `count`: si es menor que filas, hay NaNs.
- **Lectura trading (Bitcoin):** Rango de `Close` te sitúa en qué época/precio está la muestra.


In [ ]:
display(dataset.describe())


**Después de ejecutar:**

- Compara `count` con el número de filas. Anota min/max de `Close`.
- *ML:* NaNs o outliers extremos pueden distorsionar indicadores y modelos.


> **Actividad A1 (3 min):** *¿Por qué no entrenar con toda la serie desde 2012 en Colab free?* (memoria, tiempo, cambio de régimen).


---
## ⏱ Bloque B · Preparación (~50 min)
<a id='3'></a>
# 4. Preparación de datos
<a id='3.1'></a>
## 4.1 Limpieza de datos


**Paso B1.1 — Detectar valores faltantes**

- **Qué hace:** Pregunta si existe algún NaN en la tabla.
- **Qué mirar en la salida:** `Null Values = False` es lo ideal antes de indicadores.
- **Lectura trading (Bitcoin):** Huecos en precios generan señales falsas si no se tratan.


In [ ]:
print('¿Hay NaNs?', dataset.isnull().values.any())
print('NaNs por columna:\n', dataset.isnull().sum())


**Paso B1.2 — Forward fill**

- **Qué hace:** Rellena NaNs con el último valor observado (fin-ml).
- **Qué mirar en la salida:** Repite el chequeo mental: menos NaNs tras ejecutar.
- **Lectura trading (Bitcoin):** Asume que el último precio sigue vigente en el hueco — razonable en minutos, dudoso en días.


In [ ]:
dataset[dataset.columns.values] = dataset[dataset.columns.values].ffill()
print('¿NaNs tras ffill?', dataset.isnull().values.any())


**Paso B1.3 — Quitar marca de tiempo**

- **Qué hace:** Elimina `Timestamp`; el orden de filas ya indexa el tiempo.
- **Qué mirar en la salida:** Columna `Timestamp` ya no está en `dataset.columns`.
- **Lectura ML:** El modelo no usa la fecha cruda; solo features numéricas.


In [ ]:
dataset = dataset.drop(columns=['Timestamp'])
print('Columnas:', list(dataset.columns))


<a id='3.2'></a>
## 4.2 Datos categóricos (plantilla maestra)

En el master template (crédito alemán) se codifican variables categóricas. **En Bitcoin todas las entradas son numéricas** (OHLCV); no hay one-hot encoding. Si añadieras calendario (día de la semana), ahí aplicarías la sección 4.2 del template.


<a id='3.3'></a>
## 4.3 Preparar datos para clasificación

### Paso a paso — construcción de la etiqueta (aplicación)

1. **Elegir ventanas:** 10 min (corto) y 60 min (largo).
2. **Calcular SMA** sobre `Close`.
3. **Comparar:** corto > largo → etiqueta **1** (régimen alcista de corto plazo).
4. **Operativa:** 1 ≈ “estar largo”; 0 ≈ “no largo” (sin short explícito en el caso base).

$$\text{signal}_t = \mathbb{1}\{\text{SMA}_{10}(P)_t > \text{SMA}_{60}(P)_t\}$$

Las SMA usadas para etiquetar se **eliminan** de $X$ antes de entrenar para evitar copiar la regla trivialmente.


**Paso B3.1 — SMA corta (10 min)**

- **Qué hace:** Calcula media móvil simple de `Close` con ventana 10.
- **Qué mirar en la salida:** `short_mavg` cerca de `Close` pero más suave.
- **Lectura trading (Bitcoin):** Captura tendencia muy reciente (≈10 minutos).


In [ ]:
dataset['short_mavg'] = dataset['Close'].rolling(window=10, min_periods=1, center=False).mean()
dataset[['Close', 'short_mavg']].tail(3)


**Paso B3.2 — SMA larga (60 min)**

- **Qué hace:** Media móvil de 60 minutos sobre `Close`.
- **Qué mirar en la salida:** `long_mavg` cambia más lento que `short_mavg`.
- **Lectura trading (Bitcoin):** Proxy de tendencia de la última hora.


In [ ]:
dataset['long_mavg'] = dataset['Close'].rolling(window=60, min_periods=1, center=False).mean()
dataset[['Close', 'short_mavg', 'long_mavg']].tail(3)


**Paso B3.3 — Crear etiqueta `signal`**

- **Qué hace:** 1 si SMA corta > SMA larga; 0 en caso contrario.
- **Qué mirar en la salida:** Nueva columna `signal` con valores 0.0 y 1.0.
- **Lectura trading (Bitcoin):** 1 ≈ régimen alcista de corto plazo (estar largo en la demo).
- **Lectura ML:** Esta columna será nuestra variable objetivo $Y$.


In [ ]:
dataset['signal'] = np.where(dataset['short_mavg'] > dataset['long_mavg'], 1.0, 0.0)
dataset[['Close', 'short_mavg', 'long_mavg', 'signal']].tail(5)


**Paso B3.4 — Balance de clases**

- **Qué hace:** Cuenta proporción de 0 y 1 en `signal`.
- **Qué mirar en la salida:** Porcentajes en `value_counts(normalize=True)`.
- **Lectura trading (Bitcoin):** Muchos 1 → mercado alcista en la ventana; pocos 1 → lateral/bajista.
- **Lectura ML:** Clase muy mayoritaria infla accuracy de un modelo trivial.


In [ ]:
dataset['signal'].value_counts(normalize=True)


**Después de ejecutar:**

- Si ~90% es clase 1, un modelo que siempre predice 1 ya tiene ~90% accuracy.
- *Trading:* Por eso en long-only importan precision y falsos positivos.


> **Actividad B1 (5 min):** si la clase 1 tiene > 55 %, ¿qué accuracy obtiene un modelo que *siempre* predice 1? (Respuesta: ~proporción de unos.) Por eso miramos precision/recall en trading.



<a id='3.4'></a>
## 4.4 Ingeniería de características — indicadores técnicos

### Paso a paso — para qué sirve cada familia (Bitcoin)

| Familia | Variables (ej.) | Lectura en trading |
|---------|-----------------|---------------------|
| **Tendencia** | EMA10/30/200, MA21/63/252 | Precio vs media: ¿vamos arriba o abajo del consenso? |
| **Momentum** | MOM10/30, ROC10/30 | Velocidad del cambio de precio (% o diferencia) |
| **Sobrecompra/sobreventa** | RSI10/30/200 | RSI alto → muchas subidas recientes; bajo → muchas caídas |
| **Estocástico** | %K, %D | Posición del cierre dentro del rango High–Low reciente |
| **Liquidez / nivel** | `Close`, `Volume_(BTC)`, `Weighted_Price` | Contexto de precio y actividad |

A continuación: **un cálculo por celda** (mismas fórmulas que fin-ml). Lee **Qué hace / Qué mirar** antes de ejecutar.


**Paso B4.0a — Definir función EMA**

- **Qué hace:** Crea la función auxiliar `EMA(df, n)` (fin-ml).
- **Qué mirar en la salida:** Solo define la función; aún no modifica `dataset`.
- **Lectura trading (Bitcoin):** EMA pondera más los cierres recientes que una SMA.


In [ ]:
def EMA(df, n):
    EMA = pd.Series(df['Close'].ewm(span=n, min_periods=n).mean(), name='EMA_' + str(n))
    return EMA


**Paso B4.1.10 — Columna EMA10**

- **Qué hace:** Calcula y guarda `EMA10` en el dataset.
- **Qué mirar en la salida:** Último valor de EMA10 impreso.
- **Lectura trading (Bitcoin):** Tendencia muy corta (~10 min).
- **Lectura ML:** Entra luego en la matriz $X$.


In [ ]:
dataset['EMA10'] = EMA(dataset, 10)
print('EMA10 (último):', round(float(dataset['EMA10'].iloc[-1]), 2))


**Paso B4.1.30 — Columna EMA30**

- **Qué hace:** Calcula y guarda `EMA30` en el dataset.
- **Qué mirar en la salida:** Último valor de EMA30 impreso.
- **Lectura trading (Bitcoin):** Tendencia intermedia (~30 min).
- **Lectura ML:** Entra luego en la matriz $X$.


In [ ]:
dataset['EMA30'] = EMA(dataset, 30)
print('EMA30 (último):', round(float(dataset['EMA30'].iloc[-1]), 2))


**Paso B4.1.200 — Columna EMA200**

- **Qué hace:** Calcula y guarda `EMA200` en el dataset.
- **Qué mirar en la salida:** Último valor de EMA200 impreso.
- **Lectura trading (Bitcoin):** Tendencia lenta (~200 min en datos de 1 min).
- **Lectura ML:** Entra luego en la matriz $X$.


In [ ]:
dataset['EMA200'] = EMA(dataset, 200)
print('EMA200 (último):', round(float(dataset['EMA200'].iloc[-1]), 2))


**Paso B4.2a — Definir función ROC**

- **Qué hace:** Rate of change en porcentaje respecto a hace n minutos.
- **Qué mirar en la salida:** Función `ROC` definida.
- **Lectura trading (Bitcoin):** ROC > 0 → precio subió vs hace n minutos.


In [ ]:
def ROC(df, n):
    M = df.diff(n - 1)
    N = df.shift(n - 1)
    ROC = pd.Series(((M / N) * 100), name='ROC_' + str(n))
    return ROC


**Paso B4.2.10 — Columna ROC10**

- **Qué hace:** Aplica ROC al cierre con ventana 10.
- **Qué mirar en la salida:** Valor reciente de ROC10.
- **Lectura trading (Bitcoin):** Magnitud del movimiento reciente en %.


In [ ]:
dataset['ROC10'] = ROC(dataset['Close'], 10)
print('ROC10 (último):', round(float(dataset['ROC10'].iloc[-1]), 3))


**Paso B4.2.30 — Columna ROC30**

- **Qué hace:** Aplica ROC al cierre con ventana 30.
- **Qué mirar en la salida:** Valor reciente de ROC30.
- **Lectura trading (Bitcoin):** Magnitud del movimiento reciente en %.


In [ ]:
dataset['ROC30'] = ROC(dataset['Close'], 30)
print('ROC30 (último):', round(float(dataset['ROC30'].iloc[-1]), 3))


**Paso B4.3a — Definir función MOM**

- **Qué hace:** Momentum = diferencia de precio en n minutos.
- **Qué mirar en la salida:** Función `MOM` definida.
- **Lectura trading (Bitcoin):** Similar a ROC pero en unidades de precio, no %.


In [ ]:
def MOM(df, n):
    MOM = pd.Series(df.diff(n), name='Momentum_' + str(n))
    return MOM


**Paso B4.3.10 — Columna MOM10**

- **Qué hace:** Diferencia de Close vs hace 10 minutos.
- **Qué mirar en la salida:** Último MOM10.


In [ ]:
dataset['MOM10'] = MOM(dataset['Close'], 10)
print('MOM10 (último):', round(float(dataset['MOM10'].iloc[-1]), 2))


**Paso B4.3.30 — Columna MOM30**

- **Qué hace:** Diferencia de Close vs hace 30 minutos.
- **Qué mirar en la salida:** Último MOM30.


In [ ]:
dataset['MOM30'] = MOM(dataset['Close'], 30)
print('MOM30 (último):', round(float(dataset['MOM30'].iloc[-1]), 2))


**Paso B4.4a — Definir función RSI**

- **Qué hace:** Implementación RSI del repo (0–100).
- **Qué mirar en la salida:** Función `RSI` definida.
- **Lectura trading (Bitcoin):** RSI alto → dominan subidas recientes; bajo → dominan caídas.


In [ ]:
def RSI(series, period):
    delta = series.diff().dropna()
    u = delta * 0
    d = u.copy()
    u[delta > 0] = delta[delta > 0]
    d[delta < 0] = -delta[delta < 0]
    u[u.index[period - 1]] = np.mean(u[:period])
    u = u.drop(u.index[:(period - 1)])
    d[d.index[period - 1]] = np.mean(d[:period])
    d = d.drop(d.index[:(period - 1)])
    rs = u.ewm(com=period - 1, adjust=False).mean() / d.ewm(com=period - 1, adjust=False).mean()
    return 100 - 100 / (1 + rs)


**Paso B4.4.10 — Columna RSI10**

- **Qué hace:** RSI del cierre con periodo 10.
- **Qué mirar en la salida:** RSI10 entre 0 y 100 (último valor).


In [ ]:
dataset['RSI10'] = RSI(dataset['Close'], 10)
print('RSI10 (último):', round(float(dataset['RSI10'].iloc[-1]), 2))


**Paso B4.4.30 — Columna RSI30**

- **Qué hace:** RSI del cierre con periodo 30.
- **Qué mirar en la salida:** RSI30 entre 0 y 100 (último valor).


In [ ]:
dataset['RSI30'] = RSI(dataset['Close'], 30)
print('RSI30 (último):', round(float(dataset['RSI30'].iloc[-1]), 2))


**Paso B4.4.200 — Columna RSI200**

- **Qué hace:** RSI del cierre con periodo 200.
- **Qué mirar en la salida:** RSI200 entre 0 y 100 (último valor).


In [ ]:
dataset['RSI200'] = RSI(dataset['Close'], 200)
print('RSI200 (último):', round(float(dataset['RSI200'].iloc[-1]), 2))


**Paso B4.5a — Funciones estocástico %K y %D**

- **Qué hace:** Posición del cierre en el rango High–Low reciente.
- **Qué mirar en la salida:** Funciones `STOK` y `STOD` definidas.
- **Lectura trading (Bitcoin):** %K reacciona rápido; %D es media móvil de %K (más suave).


In [ ]:
def STOK(close, low, high, n):
    STOK = ((close - low.rolling(n).min()) / (high.rolling(n).max() - low.rolling(n).min())) * 100
    return STOK

def STOD(close, low, high, n):
    STOK = ((close - low.rolling(n).min()) / (high.rolling(n).max() - low.rolling(n).min())) * 100
    STOD = STOK.rolling(3).mean()
    return STOD


**Paso B4.5.10K — %K10**

- **Qué hace:** Estocástico %K con ventana 10.
- **Qué mirar en la salida:** Último %K10 (0–100).


In [ ]:
dataset['%K10'] = STOK(dataset['Close'], dataset['Low'], dataset['High'], 10)
print('%K10 (último):', round(float(dataset['%K10'].iloc[-1]), 2))


**Paso B4.5.10D — %D10**

- **Qué hace:** Suavizado %D (media de %K, ventana 3).
- **Qué mirar en la salida:** Último %D10.


In [ ]:
dataset['%D10'] = STOD(dataset['Close'], dataset['Low'], dataset['High'], 10)
print('%D10 (último):', round(float(dataset['%D10'].iloc[-1]), 2))


**Paso B4.5.30K — %K30**

- **Qué hace:** Estocástico %K con ventana 30.
- **Qué mirar en la salida:** Último %K30 (0–100).


In [ ]:
dataset['%K30'] = STOK(dataset['Close'], dataset['Low'], dataset['High'], 30)
print('%K30 (último):', round(float(dataset['%K30'].iloc[-1]), 2))


**Paso B4.5.30D — %D30**

- **Qué hace:** Suavizado %D (media de %K, ventana 3).
- **Qué mirar en la salida:** Último %D30.


In [ ]:
dataset['%D30'] = STOD(dataset['Close'], dataset['Low'], dataset['High'], 30)
print('%D30 (último):', round(float(dataset['%D30'].iloc[-1]), 2))


**Paso B4.5.200K — %K200**

- **Qué hace:** Estocástico %K con ventana 200.
- **Qué mirar en la salida:** Último %K200 (0–100).


In [ ]:
dataset['%K200'] = STOK(dataset['Close'], dataset['Low'], dataset['High'], 200)
print('%K200 (último):', round(float(dataset['%K200'].iloc[-1]), 2))


**Paso B4.5.200D — %D200**

- **Qué hace:** Suavizado %D (media de %K, ventana 3).
- **Qué mirar en la salida:** Último %D200.


In [ ]:
dataset['%D200'] = STOD(dataset['Close'], dataset['Low'], dataset['High'], 200)
print('%D200 (último):', round(float(dataset['%D200'].iloc[-1]), 2))


**Paso B4.6a — Definir función MA**

- **Qué hace:** Media móvil simple con `min_periods=n` (fin-ml).
- **Qué mirar en la salida:** Función `MA` definida.


In [ ]:
def MA(df, n):
    MA = pd.Series(df['Close'].rolling(n, min_periods=n).mean(), name='MA_' + str(n))
    return MA


**Paso B4.6.MA21 — Columna MA21**

- **Qué hace:** MA simple con ventana 10 minutos.
- **Qué mirar en la salida:** Último valor de MA21.
- **Lectura trading (Bitcoin):** Nombre MA21 en repo; ventana 10 min.


In [ ]:
dataset['MA21'] = MA(dataset, 10)
print('MA21 (último):', round(float(dataset['MA21'].iloc[-1]), 2))


**Paso B4.6.MA63 — Columna MA63**

- **Qué hace:** MA simple con ventana 30 minutos.
- **Qué mirar en la salida:** Último valor de MA63.
- **Lectura trading (Bitcoin):** Nombre MA63; ventana 30 min.


In [ ]:
dataset['MA63'] = MA(dataset, 30)
print('MA63 (último):', round(float(dataset['MA63'].iloc[-1]), 2))


**Paso B4.6.MA252 — Columna MA252**

- **Qué hace:** MA simple con ventana 200 minutos.
- **Qué mirar en la salida:** Último valor de MA252.
- **Lectura trading (Bitcoin):** Nombre MA252; ventana 200 min.


In [ ]:
dataset['MA252'] = MA(dataset, 200)
print('MA252 (último):', round(float(dataset['MA252'].iloc[-1]), 2))


**Paso B4.7 — Revisión del dataset enriquecido**

- **Qué hace:** Muestra las últimas filas con precio, señal e indicadores.
- **Qué mirar en la salida:** Muchas columnas; verifica que no haya NaN al final.
- **Lectura ML:** Tras `dropna` perderemos filas iniciales con ventanas incompletas.


In [ ]:
print('Número de columnas:', len(dataset.columns))
dataset.tail(3)


<a id='3.5'></a>
## 4.5 Visualización de datos

Exploramos relaciones lineales entre indicadores (como en el caso Bitcoin del repo).


**Paso B5.1 — Eliminar columnas no predictivas**

- **Qué hace:** Quita OHLC crudo, volumen fiat y SMA de etiquetado.
- **Qué mirar en la salida:** Lista de columnas restantes (incluye `signal`).
- **Lectura trading (Bitcoin):** Mantenemos `Close`, volumen BTC y features derivados.
- **Lectura ML:** Evita que el modelo copie trivialmente la regla SMA vía `short_mavg`.


In [ ]:
dataset = dataset.drop(['High', 'Low', 'Open', 'Volume_(Currency)', 'short_mavg', 'long_mavg'], axis=1)
print('Columnas tras drop:', list(dataset.columns))


**Paso B5.2 — Eliminar filas con NaN**

- **Qué hace:** Borra filas donde algún indicador aún no está definido.
- **Qué mirar en la salida:** Cuántas filas se eliminaron vs antes.
- **Lectura ML:** Primeras filas tras ventanas largas (p. ej. 200 min) suelen ser NaN.


In [ ]:
filas_antes = len(dataset)
dataset = dataset.dropna(axis=0)
print('Filas eliminadas por NaN:', filas_antes - len(dataset))
print('Filas listas para modelado:', len(dataset))


**Paso B5.3a — Matriz de correlación (números)**

- **Qué hace:** Calcula correlación lineal entre todas las columnas numéricas.
- **Qué mirar en la salida:** Objeto `correlation` (usa la celda siguiente para graficar).
- **Lectura ML:** Correlaciones |r|>0.9 sugieren redundancia entre features.


In [ ]:
correlation = dataset.corr()
print('Tamaño matriz:', correlation.shape)


**Paso B5.3b — Heatmap de correlación**

- **Qué hace:** Visualiza la matriz (como el caso Bitcoin del repo).
- **Qué mirar en la salida:** Bloques rojos/azules = variables que se mueven juntas.
- **Lectura trading (Bitcoin):** EMA/MA/EMA200 a menudo correlacionan fuerte con Close.
- **Lectura ML:** LDA/LR sufren con colinealidad; RF/GBM la toleran mejor.


In [ ]:
figsize = (10, 10) if MODO_CLASE_3H else (15, 15)
plt.figure(figsize=figsize)
plt.title('Matriz de correlación')
sns.heatmap(correlation, vmax=1, square=True, annot=False, cmap='cubehelix')
plt.show()


<a id='3.6'></a>
## 4.6 Selección de variables (plantilla maestra)

El master template usa filtros y dominio del negocio. Aquí **mantenemos el set del repo** (precio, volumen, indicadores) y excluimos solo columnas crudas ya reemplazadas por features. La variable objetivo es `signal`.


<a id='3.7'></a>
## 4.7 Transformación — estandarización

En el libro, **StandardScaler** (media 0, varianza 1) ayuda a modelos sensibles a escala (KNN, redes, SVM). En el notebook original se usa al **Grid Search** del Random Forest; el modelo final se entrena sobre `X_train` sin escalar — replicamos ese comportamiento.


---
## ⏱ Bloque C · Modelos (~40 min)
<a id='4'></a>
# 5. Evaluar algoritmos y modelos

### Vista unificada: del precio Bitcoin a la predicción

```
Precio minuto a minuto → indicadores X_t → modelo → ŷ_t (0/1)
                                              ↓
                         comparar con signal_t (SMA) en validación
                                              ↓
                         backtest: retorno × ŷ_{t-1}
```

<a id='4.1'></a>
## 5.1 Partición entrenamiento / validación

### Paso a paso (ML)

1. Tomar las **últimas `N_FILAS`** (mercado reciente).
2. Separar **Y** = `signal`, **X** = resto de columnas numéricas.
3. **80 % train** → ajustar modelos; **20 % validation** → simular “futuro” no visto en el ajuste.
4. `stratify=Y` mantiene proporción de 0/1 en ambos conjuntos.

Misma lógica que fin-ml: `random_state=1`.


**Paso C1.1 — Recortar ventana temporal**

- **Qué hace:** Toma las últimas `N_FILAS` filas del dataset limpio.
- **Qué mirar en la salida:** Valor de `n_use` impreso.
- **Lectura trading (Bitcoin):** Enfoca el modelo en el tramo más reciente de la muestra.


In [ ]:
n_use = min(N_FILAS, len(dataset))
subset_dataset = dataset.iloc[-n_use:]
print('Filas para modelado:', n_use)


**Paso C1.2 — Variable objetivo Y**

- **Qué hace:** Extrae la columna `signal` como etiqueta.
- **Qué mirar en la salida:** Tamaño de `Y` y media (proporción de unos).
- **Lectura ML:** $Y \in \{0,1\}$ — clasificación binaria.


In [ ]:
Y = subset_dataset['signal']
print('Y shape:', Y.shape, '| P(Y=1):', round(float(Y.mean()), 3))


**Paso C1.3 — Matriz de features X**

- **Qué hace:** Todas las columnas excepto `signal`.
- **Qué mirar en la salida:** Número de features y filas en `X`.
- **Lectura ML:** Cada fila es un minuto; cada columna un indicador o precio/volumen.


In [ ]:
X = subset_dataset.loc[:, subset_dataset.columns != 'signal']
print('X shape:', X.shape)
print('Features:', list(X.columns))


**Paso C1.4 — Partición train / validation**

- **Qué hace:** 80% train, 20% validation (`random_state=1`, fin-ml).
- **Qué mirar en la salida:** Shapes de train y validation.
- **Lectura ML:** Validation simula datos no usados al ajustar hiperparámetros finales.


In [ ]:
validation_size = 0.2
X_train, X_validation, Y_train, Y_validation = train_test_split(
    X, Y, test_size=validation_size, random_state=1
)
print('Train:', X_train.shape, 'Validation:', X_validation.shape)


**Después de ejecutar:**

- Comprueba que la proporción de Y=1 sea similar en train y validation.
- *ML:* No es split temporal estricto; es el mismo esquema que fin-ml para comparar.


<a id='4.2'></a>
## 5.2 Opciones de prueba y métricas de evaluación


**Paso C2.1 — Número de folds**

- **Qué hace:** Copia `N_FOLDS` de la configuración de sesión.
- **Qué mirar en la salida:** `num_folds` (3 en clase, 10 en modo completo).


In [ ]:
num_folds = N_FOLDS
print('Folds CV:', num_folds)


**Paso C2.2 — Semilla de CV**

- **Qué hace:** Fija aleatoriedad del KFold para resultados reproducibles.
- **Qué mirar en la salida:** `seed` = CV_SEED.


In [ ]:
seed = CV_SEED
print('Seed CV:', seed)


**Paso C2.3 — Métrica de scoring**

- **Qué hace:** Define qué optimiza CV y Grid Search.
- **Qué mirar en la salida:** Variable `scoring` (por defecto `accuracy`).
- **Lectura trading (Bitcoin):** Prueba `'precision'` o `'recall'` en Actividad C1 (long-only).


In [ ]:
scoring = 'accuracy'
# En clase: descomenta UNA línea y comenta accuracy
# scoring = 'precision'
# scoring = 'recall'
print('Scoring activo:', scoring)


**Métricas (libro, cap. 6):**

- **Accuracy:** $\frac{TP+TN}{N}$ — útil si clases balanceadas.
- **Precision (clase 1):** $\frac{TP}{TP+FP}$ — penaliza falsas compras.
- **Recall (clase 1):** $\frac{TP}{TP+FN}$ — penaliza perder subidas.
- **AUC-ROC:** calidad del ranking de probabilidades.

Para estrategias **long-only**, a menudo se discute precision vs recall (ver conclusión del notebook original).

### Paso a paso — validación cruzada (qué hace el código después)

1. Partir **train** en `N_FOLDS` bloques temporales aleatorios (KFold).
2. En cada fold: entrenar en k−1 bloques, medir accuracy en el bloque restante.
3. Promediar → estimación de error **sin usar validation**.
4. Repetir para cada algoritmo → **boxplot** al final del bloque C.


<a id='4.3'></a>
## 5.3 Guía de modelos e hiperparámetros

### Paso a paso — algoritmos en **modo clase** (qué aprende cada uno)

| Modelo | Entrada | Salida | En Bitcoin, intuitivamente… |
|--------|---------|--------|-----------------------------|
| **LR** | Vector $X_t$ | $P(\text{compra})$ | Combinación lineal de RSI, ROC… → probabilidad de régimen alcista |
| **LDA** | $X_t$ | Clase 0/1 | Frontera lineal; baseline rápido |
| **CART** | $X_t$ | 0/1 | Reglas del tipo “si RSI < 30 y ROC > 0 → compra” |
| **GBM** | $X_t$ | 0/1 | Corrige errores de árboles pequeños secuencialmente |
| **RF** | $X_t$ | 0/1 | Muchos árboles en submuestras → robustez al ruido minuto a minuto |

En **modo completo** (`MODO_CLASE_3H = False`) se añaden KNN, NB, NN, AdaBoost como en fin-ml.

| Código | Modelo | Idea (Blueprints) | Hiperparámetro clave | Si lo subes demasiado… |
|--------|--------|-------------------|----------------------|-------------------------|
| LR | Regresión logística | $P(y=1|x)=\sigma(w^\top x+b)$ | `C` (regularización) | Sobreajuste, coeficientes explosivos |
| LDA | Discriminante lineal | Frontera lineal, covarianza compartida | — | Modelo rígido |
| KNN | k-vecinos | Voto local | `n_neighbors` | Frontera ruidosa si k es pequeño |
| CART | Árbol | Particiones por impureza (Gini/entropía) | `max_depth` | Memoriza ruido minuto a minuto |
| NB | Naive Bayes | Independencia condicional | — | Sesgo si features correlacionadas |
| NN | MLP | Capas ocultas no lineales | `hidden_layer_sizes`, `alpha` | Más capacidad → más riesgo de sobreajuste |
| AB / GBM | Boosting | Suma de clasificadores débiles | `n_estimators`, `learning_rate` | Mejora CV hasta plateau |
| RF | Random Forest | Bagging de árboles | `n_estimators`, `max_depth`, `criterion` | Más árboles → menor varianza |

**Demostración visual** (dos features para poder graficar): efecto de `max_depth` (árbol) y `C` (logística).


> **Pausa docente (2 min):** enseñar un gráfico train vs CV antes de seguir al benchmark.


**Paso C3.0 — Submuestra 2D para gráficos**

- **Qué hace:** Solo RSI10 y ROC10 + signal (visualización de hiperparámetros).
- **Qué mirar en la salida:** Tamaño de `Xd_tr`.
- **Lectura ML:** Reducir a 2 features permite interpretar curvas de validación.


In [ ]:
from sklearn.pipeline import Pipeline

demo_cols = ['RSI10', 'ROC10']
demo = subset_dataset[demo_cols + ['signal']].dropna()
Xd = demo[demo_cols]
yd = demo['signal']
Xd_tr, _, yd_tr, _ = train_test_split(Xd, yd, test_size=0.3, random_state=1, stratify=yd)
print('Demo 2D — filas train:', len(Xd_tr))


**Paso C3.1 — Curva de validación — árbol (max_depth)**

- **Qué hace:** Entrena árboles con distintas profundidades; compara train vs CV.
- **Qué mirar en la salida:** Gráfico: si train sube y CV baja → sobreajuste.
- **Lectura trading (Bitcoin):** Profundidad alta = reglas muy específicas al ruido del minuto.


In [ ]:
depths = [2, 4, 6, 8, 10, 15] if MODO_CLASE_3H else [2, 3, 4, 5, 6, 8, 10, 15, 20]
pipe = Pipeline([('sc', StandardScaler()), ('clf', DecisionTreeClassifier(random_state=1))])
tr, va = validation_curve(pipe, Xd_tr, yd_tr, param_name='clf__max_depth', param_range=depths, cv=3, scoring='accuracy')
plt.figure(figsize=(8, 4))
plt.plot(depths, tr.mean(1), 'o-', label='Train')
plt.plot(depths, va.mean(1), 'o-', label='CV')
plt.xlabel('max_depth'); plt.ylabel('Accuracy'); plt.title('Árbol: profundidad vs desempeño')
plt.legend(); plt.show()


**Paso C3.2 — Curva de validación — logística (C)**

- **Qué hace:** Varía regularización `C` en escala logarítmica.
- **Qué mirar en la salida:** Gráfico semilogx: busca C donde CV sea alta y train no dispare.
- **Lectura ML:** C grande → modelo más flexible; C pequeño → más regularización.


In [ ]:
Cs = np.logspace(-3, 2, 8)
pipe_lr = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=3000, random_state=1))])
tr, va = validation_curve(pipe_lr, Xd_tr, yd_tr, param_name='clf__C', param_range=Cs, cv=3, scoring='accuracy')
plt.figure(figsize=(8, 4))
plt.semilogx(Cs, tr.mean(1), 'o-', label='Train')
plt.semilogx(Cs, va.mean(1), 'o-', label='CV')
plt.xlabel('C'); plt.title('Logística: regularización'); plt.legend(); plt.show()


**Cómo leer los gráficos:** si la curva de *train* sube y la de *CV* cae, estás **sobreajustando** (modelo memoriza). Elige hiperparámetros donde CV esté alta y la brecha train–CV sea moderada.


<a id='4.4'></a>
## 5.4 Comparar modelos

### Paso a paso — lectura del benchmark

1. Ejecutar la celda que define la **lista `models`**.
2. Ejecutar el **bucle CV**: imprime `media (desv. estándar)` por modelo.
3. Ejecutar el **boxplot**: cada caja = distribución de accuracy en folds.
4. **Elegir candidato** para Grid Search (en el libro: Random Forest).

- **Modo clase:** 5 modelos (~8–12 min Colab).
- **Modo completo:** 9 modelos (fin-ml).


**Paso C4.1 — Inicializar benchmark CV**

- **Qué hace:** Crea listas vacías y un KFold compartido para todos los modelos.
- **Qué mirar en la salida:** Mensaje con número de folds.


In [ ]:
results = []
names = []
kfold = KFold(n_splits=num_folds, shuffle=True, random_state=seed)
print('Benchmark CV — folds:', num_folds, '| scoring:', scoring)


**Paso C4.2 — Lista de modelos (modo sesión)**

- **Qué hace:** Muestra qué algoritmos se evaluarán según `MODO_CLASE_3H`.
- **Qué mirar en la salida:** Lista impresa de códigos LR, LDA, …


In [ ]:
if MODO_CLASE_3H:
    bench_list = [
        ('LR', LogisticRegression(max_iter=3000)),
        ('LDA', LinearDiscriminantAnalysis()),
        ('CART', DecisionTreeClassifier(max_depth=12, random_state=1)),
        ('GBM', GradientBoostingClassifier(n_estimators=50, random_state=1)),
        ('RF', RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=1)),
    ]
else:
    bench_list = [
        ('LR', LogisticRegression(max_iter=3000)),
        ('LDA', LinearDiscriminantAnalysis()),
        ('KNN', KNeighborsClassifier()),
        ('CART', DecisionTreeClassifier()),
        ('NB', GaussianNB()),
        ('NN', MLPClassifier(max_iter=400)),
        ('AB', AdaBoostClassifier()),
        ('GBM', GradientBoostingClassifier()),
        ('RF', RandomForestClassifier(n_jobs=-1)),
    ]
print('Modelos:', [b[0] for b in bench_list])


**Paso C4.3-LR — CV — LR**

- **Qué hace:** Un fold a la vez: entrena **LR** en k−1 particiones y mide la métrica `scoring` en la restante.
- **Qué mirar en la salida:** Línea `LR: media (std)` — anota la media.
- **Lectura trading (Bitcoin):** P(compra) ~ w·indicadores.
- **Lectura ML:** Frontera lineal; baseline.


In [ ]:
_found = False
for _code, _model in bench_list:
    if _code == 'LR':
        cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
        results.append(cv_results)
        names.append('LR')
        print('LR: %f (%f)' % (cv_results.mean(), cv_results.std()))
        _found = True
        break
if not _found:
    print('LR: no incluido en bench_list (omitido).')


**Paso C4.3-LDA — CV — LDA**

- **Qué hace:** Un fold a la vez: entrena **LDA** en k−1 particiones y mide la métrica `scoring` en la restante.
- **Qué mirar en la salida:** Línea `LDA: media (std)` — anota la media.
- **Lectura trading (Bitcoin):** Baseline rápido en alta dimensión.
- **Lectura ML:** Frontera lineal LDA.


In [ ]:
_found = False
for _code, _model in bench_list:
    if _code == 'LDA':
        cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
        results.append(cv_results)
        names.append('LDA')
        print('LDA: %f (%f)' % (cv_results.mean(), cv_results.std()))
        _found = True
        break
if not _found:
    print('LDA: no incluido en bench_list (omitido).')


**Paso C4.3-CART — CV — CART**

- **Qué hace:** Un fold a la vez: entrena **CART** en k−1 particiones y mide la métrica `scoring` en la restante.
- **Qué mirar en la salida:** Línea `CART: media (std)` — anota la media.
- **Lectura trading (Bitcoin):** Reglas locales minuto a minuto.
- **Lectura ML:** Árbol prof. limitada 12.


In [ ]:
_found = False
for _code, _model in bench_list:
    if _code == 'CART':
        cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
        results.append(cv_results)
        names.append('CART')
        print('CART: %f (%f)' % (cv_results.mean(), cv_results.std()))
        _found = True
        break
if not _found:
    print('CART: no incluido en bench_list (omitido).')


**Paso C4.3-GBM — CV — GBM**

- **Qué hace:** Un fold a la vez: entrena **GBM** en k−1 particiones y mide la métrica `scoring` en la restante.
- **Qué mirar en la salida:** Línea `GBM: media (std)` — anota la media.
- **Lectura trading (Bitcoin):** Corrige errores de modelos previos.
- **Lectura ML:** 50 árboles débiles secuenciales.


In [ ]:
_found = False
for _code, _model in bench_list:
    if _code == 'GBM':
        cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
        results.append(cv_results)
        names.append('GBM')
        print('GBM: %f (%f)' % (cv_results.mean(), cv_results.std()))
        _found = True
        break
if not _found:
    print('GBM: no incluido en bench_list (omitido).')


**Paso C4.3-RF — CV — RF**

- **Qué hace:** Un fold a la vez: entrena **RF** en k−1 particiones y mide la métrica `scoring` en la restante.
- **Qué mirar en la salida:** Línea `RF: media (std)` — anota la media.
- **Lectura trading (Bitcoin):** Robusto al ruido; suele ganar el benchmark.
- **Lectura ML:** 50 árboles en paralelo.


In [ ]:
_found = False
for _code, _model in bench_list:
    if _code == 'RF':
        cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
        results.append(cv_results)
        names.append('RF')
        print('RF: %f (%f)' % (cv_results.mean(), cv_results.std()))
        _found = True
        break
if not _found:
    print('RF: no incluido en bench_list (omitido).')


**Paso C4.3-KNN — CV — KNN (solo modo completo)**

- **Qué hace:** Evalúa **KNN** cuando `MODO_CLASE_3H = False`.
- **Qué mirar en la salida:** Si estás en modo 3 h, verás mensaje de omitido.
- **Lectura trading (Bitcoin):** Vecinos cercanos en RSI/EMA/etc.
- **Lectura ML:** Clasificador por distancia en el espacio de features.


In [ ]:
if MODO_CLASE_3H:
    print('KNN: omitido en MODO_CLASE_3H')
else:
    for _code, _model in bench_list:
        if _code == 'KNN':
            cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
            results.append(cv_results)
            names.append('KNN')
            print('KNN: %f (%f)' % (cv_results.mean(), cv_results.std()))
            break


**Paso C4.3-NB — CV — NB (solo modo completo)**

- **Qué hace:** Evalúa **NB** cuando `MODO_CLASE_3H = False`.
- **Qué mirar en la salida:** Si estás en modo 3 h, verás mensaje de omitido.
- **Lectura trading (Bitcoin):** Baseline probabilístico rápido.
- **Lectura ML:** Generativo con supuesto de independencia.


In [ ]:
if MODO_CLASE_3H:
    print('NB: omitido en MODO_CLASE_3H')
else:
    for _code, _model in bench_list:
        if _code == 'NB':
            cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
            results.append(cv_results)
            names.append('NB')
            print('NB: %f (%f)' % (cv_results.mean(), cv_results.std()))
            break


**Paso C4.3-NN — CV — NN (solo modo completo)**

- **Qué hace:** Evalúa **NN** cuando `MODO_CLASE_3H = False`.
- **Qué mirar en la salida:** Si estás en modo 3 h, verás mensaje de omitido.
- **Lectura trading (Bitcoin):** Captura no linealidades.
- **Lectura ML:** MLP con capas ocultas.


In [ ]:
if MODO_CLASE_3H:
    print('NN: omitido en MODO_CLASE_3H')
else:
    for _code, _model in bench_list:
        if _code == 'NN':
            cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
            results.append(cv_results)
            names.append('NN')
            print('NN: %f (%f)' % (cv_results.mean(), cv_results.std()))
            break


**Paso C4.3-AB — CV — AB (solo modo completo)**

- **Qué hace:** Evalúa **AB** cuando `MODO_CLASE_3H = False`.
- **Qué mirar en la salida:** Si estás en modo 3 h, verás mensaje de omitido.
- **Lectura trading (Bitcoin):** Secuencia de modelos correctoras.
- **Lectura ML:** AdaBoost sobre clasificadores débiles.


In [ ]:
if MODO_CLASE_3H:
    print('AB: omitido en MODO_CLASE_3H')
else:
    for _code, _model in bench_list:
        if _code == 'AB':
            cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
            results.append(cv_results)
            names.append('AB')
            print('AB: %f (%f)' % (cv_results.mean(), cv_results.std()))
            break


**Paso C4.4 — Boxplot del benchmark**

- **Qué hace:** Compara distribución de scores CV entre algoritmos.
- **Qué mirar en la salida:** Cajas estrechas = desempeño estable entre folds.
- **Lectura ML:** Elige el ganador (suele ser RF/GBM) para el Grid Search.


In [ ]:
fig = plt.figure()
fig.suptitle('Comparación de algoritmos')
ax = fig.add_subplot(111)
plt.boxplot(results, labels=names)
ax.set_ylabel(scoring)
fig.set_size_inches(15, 8)
plt.xticks(rotation=45)
plt.show()


**Interpretación (10 min discusión):** ¿Cuál gana en **media CV**? ¿La caja es estrecha (estable)? En clase, el ganador suele ser **RF o GBM**; anoten el nombre para el grid.

> **Actividad C1 (5 min):** cambien `scoring` a `'recall'` y vuelvan a correr **solo** la celda del benchmark. ¿Cambia el ranking? ¿Por qué importa para una estrategia long?


---
## ⏱ Bloque D · Grid y modelo final (~40 min)
<a id='5'></a>
# 6. Ajuste fino y Grid Search

### Paso a paso — Grid Search (modelos)

1. **Elegir familia:** Random Forest (suele ganar en el caso del libro).
2. **Definir malla:** `n_estimators`, `max_depth`, `criterion` (Gini vs entropía).
3. **Escalar X_train** con `StandardScaler` (como fin-ml en esta etapa).
4. **GridSearchCV** prueba cada combinación × cada fold → guarda `best_params_`.
5. **Interpretar heatmap:** celdas similares = zona robusta; pico aislado = cuidado.

### Paso a paso — hiperparámetros RF en lenguaje Bitcoin

- **`n_estimators`:** más árboles → menos varianza, más tiempo de cómputo.
- **`max_depth`:** árboles más profundos → captan micro-patrones del minuto (riesgo de sobreajuste).
- **`criterion`:** forma de medir “pureza” de cada split (Gini vs entropía); impacto suele ser menor que la profundidad.


**Paso D1 — Ajustar StandardScaler**

- **Qué hace:** Calcula media/desvío de cada feature en train y transforma X_train.
- **Qué mirar en la salida:** Media ~0 en la primera feature tras escalar.
- **Lectura ML:** Solo para Grid Search RF en fin-ml; modelo final no usa escala.


In [ ]:
scaler = StandardScaler().fit(X_train)
rescaledX = scaler.transform(X_train)
print('Media tras escalar (1ª feature, muestra):', rescaledX[:, 0].mean().round(4))


**Paso D2a — Hiperparámetro n_estimators**

- **Qué hace:** Lista de candidatos de número de árboles (según modo clase).
- **Qué mirar en la salida:** Vector `n_estimators` impreso.
- **Lectura trading (Bitcoin):** Más árboles → señal más estable, más tiempo de cómputo.


In [ ]:
if MODO_CLASE_3H:
    n_estimators = [20, 50]
else:
    n_estimators = [20, 80]
print('n_estimators:', n_estimators)


**Paso D2b — Hiperparámetros max_depth y criterion**

- **Qué hace:** Profundidad máxima de cada árbol y criterio de impureza (Gini/entropía).
- **Qué mirar en la salida:** Listas `max_depth` y `criterion`.
- **Lectura ML:** max_depth bajo → menos sobreajuste al ruido minuto a minuto.


In [ ]:
max_depth = [5, 10]
criterion = ['gini'] if MODO_CLASE_3H else ['gini', 'entropy']
print('max_depth:', max_depth, '| criterion:', criterion)


**Paso D2c — Armar param_grid**

- **Qué hace:** Diccionario con todas las combinaciones a probar.
- **Qué mirar en la salida:** Número total de combinaciones.


In [ ]:
param_grid = dict(n_estimators=n_estimators, max_depth=max_depth, criterion=criterion)
print('Combinaciones a probar:', len(n_estimators) * len(max_depth) * len(criterion))


**Paso D3 — GridSearchCV — entrenamiento**

- **Qué hace:** Prueba cada combinación con CV sobre train escalado.
- **Qué mirar en la salida:** `best_score_` y `best_params_` impresos.
- **Lectura ML:** Elige hiperparámetros maximizando scoring en CV, no en validation.


In [ ]:
model = RandomForestClassifier(n_jobs=-1, random_state=1)
kfold = KFold(n_splits=num_folds, shuffle=True, random_state=seed)
grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring=scoring, cv=kfold, n_jobs=-1)
grid_result = grid.fit(rescaledX, Y_train)
print('Best CV score:', grid_result.best_score_.round(4))
print('Best params:', grid_result.best_params_)


**Paso D4 — Ranking de combinaciones**

- **Qué hace:** Lista cada combinación con rank, media CV y desviación.
- **Qué mirar en la salida:** Rank #1 coincide con `best_params_`.


In [ ]:
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
ranks = grid_result.cv_results_['rank_test_score']
for mean, stdev, param, rank in zip(means, stds, params, ranks):
    print('#%d %f (%f) with: %r' % (rank, mean, stdev, param))


**Paso D4b — Preparar tabla para heatmap**

- **Qué hace:** Convierte resultados del grid en DataFrame.
- **Qué mirar en la salida:** Filas = una fila por combinación probada.


In [ ]:
cv_df = pd.DataFrame(grid_result.cv_results_)
print('Filas en cv_df:', len(cv_df))


**Paso D4c — Heatmap Grid RF**

- **Qué hace:** Mapa de calor: max_depth vs n_estimators (por cada criterion).
- **Qué mirar en la salida:** Celdas similares = zona robusta de hiperparámetros.
- **Lectura trading (Bitcoin):** Optimizamos accuracy CV, no PnL — el heatmap no garantiza ganancias.


In [ ]:
for crit in criterion:
    sub = cv_df[cv_df['param_criterion'] == crit]
    pivot = sub.pivot(index='param_max_depth', columns='param_n_estimators', values='mean_test_score')
    plt.figure(figsize=(5, 4))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis')
    plt.title('Grid RF — criterion=%s' % crit)
    plt.show()


<a id='6'></a>
# 7. Finalizar el modelo


<a id='6.1'></a>
## 7.1 Resultados en el conjunto de validación

### Paso a paso — evaluación final (ML + trading)

1. **Reentrenar** RF con `best_params_` sobre todo **X_train** (sin escalar → igual que fin-ml).
2. **Predecir** en **X_validation** → vector `predictions`.
3. **Accuracy / reporte:** calidad global de clasificación.
4. **Matriz de confusión:** traducir TP/FP a “compras acertadas” vs “falsas alarmas”.
5. **Importancia de variables:** qué indicadores usa el bosque para imitar/mejorar la SMA.

Parámetros ganadores del grid; entrenamiento sobre **X_train sin escalar** (mismo código que fin-ml).


**Paso D5 — Modelo final en train**

- **Qué hace:** Instancia RandomForest con `best_params_` y ajusta en X_train **sin escalar**.
- **Qué mirar en la salida:** Mensaje `Entrenado con:` y dict de parámetros.
- **Lectura ML:** Replica fin-ml: escala solo en grid, no en fit final.


In [ ]:
# Usar mejores hiperparámetros del grid (fin-ml fija a mano; aquí tomamos el grid)
bp = grid_result.best_params_
model = RandomForestClassifier(
    criterion=bp.get('criterion', 'gini'),
    n_estimators=bp.get('n_estimators', 80),
    max_depth=bp.get('max_depth', 10),
    n_jobs=-1,
    random_state=1,
)
model.fit(X_train, Y_train)
print('Entrenado con:', bp)


**Paso D6 — Predicciones ŷ en validation**

- **Qué hace:** Aplica `predict` en X_validation.
- **Qué mirar en la salida:** Accuracy impresa vs Y_validation.
- **Lectura trading (Bitcoin):** Cada 1 predicho ≈ minuto donde el modelo recomienda estar largo.


In [ ]:
predictions = model.predict(X_validation)
print('Accuracy validation:', accuracy_score(Y_validation, predictions))


**Paso D7a — Matriz de confusión (números)**

- **Qué hace:** Cuenta TP, TN, FP, FN entre real y predicho.
- **Qué mirar en la salida:** Matriz 2×2 impresa.
- **Lectura trading (Bitcoin):** FP = falsa compra; FN = subida perdida.


In [ ]:
print(confusion_matrix(Y_validation, predictions))
print(classification_report(Y_validation, predictions))


**Paso D7b — Heatmap de confusión**

- **Qué hace:** Visualiza la misma matriz (más fácil de discutir en clase).
- **Qué mirar en la salida:** Diagonal fuerte = buen acuerdo con etiqueta SMA.


In [ ]:
df_cm = pd.DataFrame(
    confusion_matrix(Y_validation, predictions),
    columns=np.unique(Y_validation),
    index=np.unique(Y_validation),
)
df_cm.index.name = 'Actual'
df_cm.columns.name = 'Predicted'
sns.heatmap(df_cm, cmap='Blues', annot=True, annot_kws={'size': 16})
plt.show()


**Interpretación matriz de confusión:**

- **Falsos positivos (FP):** modelo dice compra y la regla SMA decía venta → operaciones perdedoras potenciales.
- **Falsos negativos (FN):** pierdes subidas.
- Si operas **long-only**, muchos FP pueden destruir el PnL aunque accuracy sea alta.


<a id='6.2'></a>
## 7.2 Intuición de variables / importancia de features


**Paso D8a — Importancias del bosque**

- **Qué hace:** Extrae `feature_importances_` del RF (suma 100% en el gráfico).
- **Qué mirar en la salida:** Series ordenada por importancia.
- **Lectura ML:** Impurity decrease — no es causalidad ni estabilidad temporal.


In [ ]:
Importance = pd.DataFrame({'Importance': model.feature_importances_ * 100}, index=X.columns)
Importance.sort_values('Importance', ascending=False).head(8)


**Paso D8b — Gráfico de barras**

- **Qué hace:** Visualiza qué indicadores usa más el modelo.
- **Qué mirar en la salida:** Barras largas = splits más frecuentes en el bosque.
- **Lectura trading (Bitcoin):** Si dominan RSI/ROC, el ML imita lógica momentum/tendencia.


In [ ]:
Importance.sort_values('Importance', axis=0, ascending=True).plot(kind='barh', color='steelblue', figsize=(8, 6))
plt.xlabel('Importancia de variables (%)')
plt.tight_layout()
plt.show()


**Interpretación:** indicadores de **momentum** (RSI, ROC, estocástico) suelen aparecer arriba si la etiqueta SMA es tendencial. No implica causalidad ni estabilidad temporal: reentrena y compara importancias en distintas ventanas (walk-forward del libro).


<a id='6.3'></a>
## 7.3 Guardar modelo para uso posterior (master template)


**Paso D9 — Persistir modelo**

- **Qué hace:** Guarda el RF entrenado con `pickle` (master template).
- **Qué mirar en la salida:** Archivo `finalized_model_bitcoin.sav` en el directorio de trabajo.
- **Lectura ML:** En producción versionaría modelo + fecha + features usados.


In [ ]:
filename = 'finalized_model_bitcoin.sav'
dump(model, open(filename, 'wb'))
print('Modelo guardado:', filename)


---
## ⏱ Bloque E · Backtesting y cierre (~25 min)
<a id='7'></a>
# 8. Backtesting

### Paso a paso — simulación (aplicación Bitcoin)

1. **`Market Returns`** = $\frac{P_t - P_{t-1}}{P_{t-1}}$ (retorno del BTC en ese minuto).
2. **`signal_actual`** = etiqueta SMA (referencia “benchmark” del libro).
3. **`signal_pred`** = predicción del Random Forest.
4. **Posición con lag:** usamos `shift(1)` → la señal de $t-1$ multiplica el retorno de $t$ (evita usar la señal del mismo minuto del retorno en esta demo).
5. **`Strategy Returns`** = retorno de mercado × señal predicha (0 o 1) → simula estar largo solo cuando el modelo dice 1.
6. **`Actual Returns`** = mismo esquema con la señal SMA → comparar ML vs regla original.
7. **Gráfico acumulado:** suma simple de retornos (didáctico; en producción usar log-returns, comisiones y slippage).

Retorno × señal con **`.shift(1)`** como en fin-ml.


**Paso E1a — Crear marco de backtest**

- **Qué hace:** DataFrame indexado como validation.
- **Qué mirar en la salida:** Tabla vacía con índice correcto.


In [ ]:
backtestdata = pd.DataFrame(index=X_validation.index)
print('Filas backtest:', len(backtestdata))


**Paso E1b — Señal predicha (ML)**

- **Qué hace:** Columna `signal_pred` = salida del Random Forest.
- **Qué mirar en la salida:** Valores 0/1 en `signal_pred`.
- **Lectura trading (Bitcoin):** 1 = minuto donde el modelo estaría largo.


In [ ]:
backtestdata['signal_pred'] = predictions
backtestdata['signal_pred'].value_counts()


**Paso E1c — Señal real (SMA)**

- **Qué hace:** Columna `signal_actual` = etiqueta de referencia del libro.
- **Qué mirar en la salida:** Compara conteos con predicción.


In [ ]:
backtestdata['signal_actual'] = Y_validation
backtestdata[['signal_pred', 'signal_actual']].head(3)


**Paso E2 — Retorno de mercado**

- **Qué hace:** `pct_change()` del Close en validation.
- **Qué mirar en la salida:** Primer valor NaN (no hay t−1).
- **Lectura trading (Bitcoin):** Retorno buy-and-hold minuto a minuto sin filtro.


In [ ]:
backtestdata['Market Returns'] = X_validation['Close'].pct_change()
backtestdata[['Market Returns']].head(3)


**Paso E3 — Retorno estrategia SMA (benchmark)**

- **Qué hace:** Market return × señal SMA en t−1 (`shift(1)`).
- **Qué mirar en la salida:** Columna `Actual Returns`.
- **Lectura trading (Bitcoin):** Simula seguir la regla SMA con un minuto de lag.


In [ ]:
backtestdata['Actual Returns'] = backtestdata['Market Returns'] * backtestdata['signal_actual'].shift(1)
backtestdata[['Market Returns', 'signal_actual', 'Actual Returns']].head(5)


**Paso E4 — Retorno estrategia ML**

- **Qué hace:** Market return × señal predicha en t−1.
- **Qué mirar en la salida:** Columna `Strategy Returns`.
- **Lectura trading (Bitcoin):** Compara PnL simple ML vs SMA en el mismo hold-out.


In [ ]:
backtestdata['Strategy Returns'] = backtestdata['Market Returns'] * backtestdata['signal_pred'].shift(1)
backtestdata = backtestdata.reset_index()
backtestdata[['signal_pred', 'Strategy Returns']].head(5)


**Paso E5a — Retorno acumulado (números)**

- **Qué hace:** Suma simple de retornos minuto a minuto (didáctico).
- **Qué mirar en la salida:** Totales acumulados impresos para ML vs SMA.
- **Lectura trading (Bitcoin):** No incluye comisiones Bitstamp ni slippage.


In [ ]:
cum = backtestdata[['Strategy Returns', 'Actual Returns']].cumsum()
print('Acumulado final ML:', round(float(cum['Strategy Returns'].iloc[-1]), 4))
print('Acumulado final SMA:', round(float(cum['Actual Returns'].iloc[-1]), 4))


**Paso E5b — Histograma de caminos acumulados**

- **Qué hace:** Distribución de retornos acumulados (exploración rápida).
- **Qué mirar en la salida:** Forma del histograma — no es walk-forward.


In [ ]:
backtestdata[['Strategy Returns', 'Actual Returns']].cumsum().hist()


**Paso E5c — Serie temporal acumulada**

- **Qué hace:** Gráfico principal: ¿ML supera a SMA en validation?
- **Qué mirar en la salida:** Curva más alta al final = mejor en esta demo simplificada.


In [ ]:
backtestdata[['Strategy Returns', 'Actual Returns']].cumsum().plot(figsize=(12, 5))
plt.title('Retornos acumulados (suma simple) — estrategia ML vs señal SMA')
plt.legend(['Estrategia ML', 'Señal SMA'])
plt.show()


### Conclusión (adaptada del libro / repo)

1. **Formulación:** convertir el objetivo de inversión en etiquetas y features es el primer paso; aquí la etiqueta sigue una regla SMA interpretable.
2. **Feature engineering:** indicadores de tendencia y momentum aumentan poder predictivo frente a usar solo precio crudo.
3. **Métricas:** accuracy o AUC son razonables en clasificación; si priorizas **long** con pocos falsos positivos, mira **precision**; si no quieres perder subidas, **recall**.
4. **Backtesting:** permite analizar riesgo/rentabilidad **antes** de arriesgar capital — sin comisiones, slippage ni tamaño de posición (limitaciones didácticas).

**Próximo paso académico:** dataset completo Kaggle, split temporal, costos de transacción y validación walk-forward del capítulo de producción.

> **Cierre (5 min):** ¿La estrategia ML supera a la señal SMA en el gráfico? ¿Qué falta para confiar en capital real?


<a id='8'></a>
# 9. Modo completo fin-ml (opcional — tarea en casa)

1. Arriba, pon **`MODO_CLASE_3H = False`** y vuelve a ejecutar **Runtime → Run all** (≈45–90 min con muestra GitHub; más con Kaggle completo).
2. Compara accuracy CV y backtest con la sesión de 3 h.
3. Entrega sugerida: captura del boxplot, `best_params_`, matriz de confusión y una página de interpretación (precision vs recall).
